In [37]:
import librosa
#from skimage import feature
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
from scipy import signal
import soundfile as sf

In [38]:
import pandas as pd

# --------------------------------------------------
# File paths
# --------------------------------------------------

main_file = "xc_BadenWurttem.csv"


# --------------------------------------------------
# Load files
# --------------------------------------------------

df = pd.read_csv(main_file)

        

In [39]:
df = df[
    df["species"].notna()
    & df["species"].ne("Soundscape")
].reset_index(drop=True)

In [40]:
df = df[
    ~df["user"].str.contains(
        "animal ecology and planning",
        case=False,
        na=False
    )
].reset_index(drop=True)

In [41]:
df = df[
    ~df["species"].str.contains(
        "unknown",
        case=False,
        na=False
    )
].reset_index(drop=True)

In [42]:
# Basic dataframe analysis

# Summary statistics for every column
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "unique_values": df.nunique(dropna=False),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

display(summary)

# Number of duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

# Unique-value counts for each column
for column in df.columns:
    print(f"\n{'=' * 60}\n{column}")
    print(f"Unique values: {df[column].nunique(dropna=False)}")
    display(df[column].value_counts(dropna=False).head(20))

# Date analysis
dates = pd.to_datetime(df["date"], errors="coerce")
print("\nDate range:")
print(f"Earliest: {dates.min()}")
print(f"Latest:   {dates.max()}")

# Duration analysis
duration_seconds = pd.to_timedelta(
    "00:" + df["duration"].astype(str),
    errors="coerce"
).dt.total_seconds()

print("\nDuration statistics in seconds:")
display(duration_seconds.describe())

# Numeric summary
print("\nNumeric columns:")
display(df.describe(include="number"))

# Categorical summary
print("\nCategorical columns:")
display(df.describe(include="object").T)

,dtype,non_null,missing,unique_values,missing_percent
recording_id,int64,4725,0,4725,0.00
mic,object,1846,2879,39,60.93
species,object,4725,0,254,0.00
country,object,4725,0,3,0.00
date,object,4725,0,1671,0.00
duration,object,4725,0,527,0.00
user,object,4725,0,101,0.00
recording_url,object,4725,0,4725,0.00


Duplicate rows: 0

recording_id
Unique values: 4725


1165902    1
167815     1
952998     1
958949     1
983747     1
983748     1
983749     1
983750     1
983751     1
215309     1
161463     1
163530     1
475225     1
631550     1
631552     1
840509     1
120963     1
952997     1
952994     1
952877     1
Name: recording_id, dtype: int64


mic
Unique values: 39


NaN                                                2879
AudioMoth 1.2.0                                     793
Sennheiser ME66                                     361
Built-in                                            204
img Stage Line DPR-2004 MIC QUAD MICROPHONE          88
Wildlife Acoustics Song Meter SM2+                   62
Dodotronic Hi-Sound Stereo Parabolic Microphone      61
Parabol-Mikrofon                                     52
Audiomoth 1.2.0                                      44
MEMS                                                 26
Zoom H2                                              26
Zoom H5                                              21
MEMS microphone                                      15
Sony PCM-A10                                         13
capsule AOM 5024                                     12
Uher M516                                             9
Clipsy EM272 XLR Microphone                           8
Hi-Sound DR                                     


species
Unique values: 254


... (13 outputs left out)


Numeric columns:


,recording_id
count,4.725000e+03
mean,5.886685e+05
std,3.040393e+05
min,3.338600e+04
25%,3.342590e+05
50%,5.589040e+05
75%,8.721940e+05
max,1.172469e+06



Categorical columns:


,count,unique,top,freq
mic,1846,38,AudioMoth 1.2.0,793
species,4725,254,Common Blackbird,370
country,4725,3,Germany,4594
date,4725,1671,2020-05-17,68
duration,4725,527,0:03,160
user,4725,101,Frank Holzapfel,2601
recording_url,4725,4725,https://xeno-canto.org/1165902,1


In [43]:
len(df)

4725

In [47]:
display(
    df.loc[duration_seconds.nlargest(30).index]
      .assign(duration_seconds=duration_seconds)
      .sort_values("duration_seconds", ascending=False)
)

,recording_id,mic,species,country,date,duration,user,recording_url,duration_seconds
3429,759534,Built-in,Common Blackbird,Germany,2022-03-27,45:04,Frank Holzapfel,https://xeno-canto.org/759534,2704.0
3529,796583,Built-in,Common Blackbird,Germany,2023-03-05,44:34,Frank Holzapfel,https://xeno-canto.org/796583,2674.0
3418,760583,Built-in,Common Blackbird,Germany,2022-03-27,41:10,Frank Holzapfel,https://xeno-canto.org/760583,2470.0
1520,315476,NaN,Rook,France,2016-05-06,38:52,brickegickel,https://xeno-canto.org/315476,2332.0
3549,786678,Built-in,Common Blackbird,Germany,2023-02-21,36:20,Frank Holzapfel,https://xeno-canto.org/786678,2180.0
3324,999410,AudioMoth 1.2.0,Common Blackbird,Germany,2025-03-29,34:47,Johannes Dag Mayer,https://xeno-canto.org/999410,2087.0
3358,929332,AudioMoth 1.2.0,Common Blackbird,Germany,2024-04-24,34:43,Johannes Dag Mayer,https://xeno-canto.org/929332,2083.0
3322,1085643,Sennheiser ME66,Common Blackbird,Germany,2026-03-06,33:37,Frank Holzapfel,https://xeno-canto.org/1085643,2017.0
3532,795170,Built-in,Common Blackbird,Germany,2023-03-05,32:54,Frank Holzapfel,https://xeno-canto.org/795170,1974.0
3424,759916,Built-in,Common Blackbird,Germany,2022-03-27,30:40,Frank Holzapfel,https://xeno-canto.org/759916,1840.0


In [ ]:
df.to_csv("xc_BadenWurttem_filtered.csv", index=False)

In [45]:
df_audiomoth= df[
    df["mic"].str.contains("audiomoth", case=False, na=False)
].reset_index(drop=True)

In [46]:
# Analyze df_audiomoth
# Basic dataframe overview
print(f"Shape: {df_audiomoth.shape}")
print(f"Columns: {list(df_audiomoth.columns)}")

# Summary statistics for every column
summary = pd.DataFrame({
    "dtype": df_audiomoth.dtypes.astype(str),
    "non_null": df_audiomoth.notna().sum(),
    "missing": df_audiomoth.isna().sum(),
    "unique_values": df_audiomoth.nunique(dropna=False),
    "missing_percent": (df_audiomoth.isna().mean() * 100).round(2)
})
display(summary)

# Duplicate rows
print(f"\nDuplicate rows: {df_audiomoth.duplicated().sum()}")

# Unique-value counts in each column
for column in df_audiomoth.columns:
    print(f"\n{'=' * 60}\n{column}")
    print(f"Unique values: {df_audiomoth[column].nunique(dropna=False)}")
    display(df_audiomoth[column].value_counts(dropna=False).head(20))

# Date analysis
dates_audiomoth = pd.to_datetime(df_audiomoth["date"], errors="coerce")
print("\nDate range:")
print(f"Earliest: {dates_audiomoth.min()}")
print(f"Latest:   {dates_audiomoth.max()}")

# Duration analysis
duration_seconds_audiomoth = pd.to_timedelta(
    "00:" + df_audiomoth["duration"].astype(str),
    errors="coerce"
).dt.total_seconds()

print("\nDuration statistics in seconds:")
display(duration_seconds_audiomoth.describe())

# Numeric summary
print("\nNumeric columns:")
display(df_audiomoth.describe(include="number"))

# Categorical summary
print("\nCategorical columns:")
display(df_audiomoth.describe(include="object").T)

Shape: (837, 8)
Columns: ['recording_id', 'mic', 'species', 'country', 'date', 'duration', 'user', 'recording_url']


,dtype,non_null,missing,unique_values,missing_percent
recording_id,int64,837,0,837,0.0
mic,object,837,0,2,0.0
species,object,837,0,123,0.0
country,object,837,0,1,0.0
date,object,837,0,333,0.0
duration,object,837,0,186,0.0
user,object,837,0,1,0.0
recording_url,object,837,0,837,0.0



Duplicate rows: 0

recording_id
Unique values: 837


991720    1
809514    1
839687    1
933810    1
933809    1
839686    1
933788    1
933787    1
933792    1
933791    1
933790    1
932226    1
932224    1
932223    1
932222    1
932225    1
872195    1
934906    1
934280    1
837383    1
Name: recording_id, dtype: int64


mic
Unique values: 2


AudioMoth 1.2.0    793
Audiomoth 1.2.0     44
Name: mic, dtype: int64

... (14 outputs left out)


Numeric columns:


,recording_id
count,8.370000e+02
mean,9.276946e+05
std,7.508220e+04
min,8.057380e+05
25%,8.721930e+05
50%,9.337880e+05
75%,9.562390e+05
max,1.158093e+06



Categorical columns:


,count,unique,top,freq
mic,837,2,AudioMoth 1.2.0,793
species,837,123,Common Blackbird,77
country,837,1,Germany,837
date,837,333,2024-01-19,17
duration,837,186,0:03,78
user,837,1,Johannes Dag Mayer,837
recording_url,837,837,https://xeno-canto.org/991720,1


In [51]:
df_audiomoth.to_csv("xc_Baden_AudioMoth.csv", index=False)